In [ ]:
%pip install -r requirements.txt

# Tech Challenge 4

O Desafio
Você e seu grupo foram contratados para atuar como Data Analysts em
uma empresa de e-commerce.

A empresa deseja compreender melhor o desempenho de seu portfólio e a
percepção dos consumidores, utilizando informações sobre:

<ul>
<li>Produtos E Categorias;</li>
<li>Preços E Descontos;</li>
<li>Avaliações E Quantidade De Reviews;</li>
<li>Descrições Dos Produtos;</li>
<li>Comentários Realizados Pelos Consumidores</li>
</ul>

A análise deverá buscar responder à seguinte questão:

Quais produtos, categorias e estratégias comerciais merecem maior atenção
e quais oportunidades podem ser identificadas a partir dos dados
disponíveis?

A liderança da empresa deseja compreender melhor a relação entre:

Preço → Desconto → Engajamento → Satisfação do Cliente

Algumas perguntas que podem orientar o desenvolvimento são:
<ul>
<li>Quais categorias e produtos apresentam maior relevância?</li>
<li>Quais produtos possuem maior engajamento dos consumidores?</li>
<li>Existe relação entre descontos e avaliações?</li>
<li>Quais categorias apresentam maior ou menor satisfação?</li>
<li>Existem produtos com alto engajamento, mas baixa avaliação?</li>
<li>Existem produtos com boa avaliação, mas pouca visibilidade?</li>
<li>Quais padrões podem ser identificados nas avaliações dos
consumidores?</li>
<li>Quais oportunidades ou riscos podem ser identificados para o negócio?</li>
</ul>

A solução deverá permitir a análise de:

Portfólio: exploração de produtos e categorias para identificar pontos fortes,
oportunidades e produtos que merecem atenção.
Preços e Descontos: análise da relação entre preço original, preço com
desconto e percentual de desconto

A equipe deverá investigar possíveis padrões relacionados ao
comportamento comercial dos produtos.
Engajamento e Satisfação - análise da relação entre:
Quantidade de Avaliações × Satisfação dos Clientes

Customer Insights: além dos indicadores quantitativos, a equipe deverá
explorar os conteúdos das avaliações dos consumidores.

A análise poderá buscar responder:
<ul>
<li>O que os clientes mais valorizam?</li>
<li>Quais problemas aparecem com maior frequência?</li>
<li>Quais padrões estão presentes em avaliações positivas?</li>
<li>Quais temas aparecem em avaliações negativas?</li>
</ul>

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords
import spacy

In [2]:
FILENAME = "data\\amazon.csv"
df = pd.read_csv(FILENAME, delimiter=",")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage: 183.3 KB


In [3]:
df.head()

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


## Data Cleaning

<ul>
<li>Padronização da Moeda</li>
<li>Padronização da porcentagem</li>
<li>Padronização do Rating</li>
<li>Padronização da contagem de ratings</li>
<li>Remover duplicados</li>
<li>Segmentar Categorias</li>
</ul>

Remoção de duplicadas

In [4]:
df = df.drop_duplicates()

In [5]:
df=df.dropna()

### Padronização do texto das reviews

In [6]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text) # Remove URLs
    text = re.sub(r"@\w+", "", text) # Remove mentions
    text = re.sub(r"[^\w\s]", " ", text) # Remove punctuation
    text = re.sub(r"\d+", "", text) # Remove numbers
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces
    return text
    

In [7]:
df = df.drop_duplicates(subset=["review_content", "review_title"])

In [8]:
df["review_content"] = df["review_content"].fillna("")
df["review_title"] = df["review_title"].fillna("")

In [9]:
apply_stopwords_cols = ["review_content", "review_title", "about_product", "product_name"]
stop_words = set(stopwords.words("english"))

for col in apply_stopwords_cols:
    df[col] = df[col].apply(
        lambda x: " ".join(
            word for word in str(x).split()
            if word not in stop_words
        )
    )

In [10]:
nlp = spacy.load("en_core_web_sm")
def lemmatize(text):
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

In [11]:
for i in range(len(df)):
    df.loc[i, "review_content"] = lemmatize(clean_text(df.iloc[i]["review_content"]))
    df.loc[i, "review_title"] = lemmatize(clean_text(df.iloc[i]["review_title"]))
    df.loc[i, "about_product"] = lemmatize(clean_text(df.iloc[i]["about_product"]))
    df.loc[i, "product_name"] = lemmatize(clean_text(df.iloc[i]["product_name"]))

In [32]:
temp_ls = df["review_content"].to_string(header=False, index=False).split()
temp_output = []

for word in temp_ls:
    
    if word not in temp_output:
        temp_output.append(word)

temp_output

['look',
 'durable',
 'charge',
 'fine',
 'toono',
 'complain',
 '...',
 'I',
 'order',
 'cable',
 'connect',
 'phone',
 'android',
 'auto',
 'car',
 'th...',
 'not',
 'quite',
 'sturdy',
 'good',
 'nice',
 'product',
 'work...',
 'long',
 'wire',
 'buy',
 'c...',
 'instead',
 'original',
 'apple',
 'work',
 'r',
 'fast',
 'ch...',
 'it',
 's',
 'like',
 'very',
 'item',
 'strong',
 'us...',
 'build',
 'quality',
 'come',
 'year',
 'warranty',
 'prod...',
 'worth',
 'money',
 'suitable',
 'purpose',
 'serve...',
 'use',
 'old',
 'pc',
 'internet',
 'try',
 'lubuntu',
 'ubu...',
 'day',
 'now',
 'the',
 'experience',
 'pretty',
 'pi...',
 'loose',
 'costly',
 'affair',
 'this',
 'wir...',
 'well',
 'power',
 'note',
 'pro',
 'm...',
 'xiomi',
 'a',
 'budget',
 'picture',
 've...',
 'check',
 'offera',
 'buying',
 'w',
 'awesome',
 'cabl...',
 'lg',
 'always',
 'correct',
 'delivery',
 'service',
 'so...',
 'connector',
 'provide',
 'boon',
 'samsung',
 'lcd',
 'mon...',
 'overall',
 'tv

In [ ]:
replacements = {
    
}

Segmentação de categorias

In [ ]:
# Segmentação de categorias
for index, row in df.iterrows():
    if row["category"] is str:
        temp = row["category"].split("|")
        for i in range(len(temp)):
            df.loc[index, [f"category_{i+1}"]] = temp[i]


Padronização de rating

In [ ]:
# Padronização de rating
df["rating"] = df["rating"].replace("|", "0")
df["rating"] = df["rating"].astype(float)

Padronização de moeda

In [ ]:
# Padronização de moeda
df["actual_price"] = (df["actual_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

df["discounted_price"] = (df["discounted_price"]
                      .str.replace("₹", "", regex=False)
                      .str.replace(",", "", regex=False)
                      .astype(float))

Padronização de Porcentagens

In [ ]:
# Padronização da porcentagem
df["discount_percentage"] = (df["discount_percentage"]
                      .str.replace("%", "", regex=False)
                      .str.replace(".", "", regex=False)
                      .astype(float))

df["discount_percentage"] = df["discount_percentage"]/100


Padronização do rating count

In [ ]:
# Padronização do rating count
df["rating_count"] = df["rating_count"].fillna("0,0")
df["rating_count"] = (df["rating_count"]
                      .str.replace(",", "", regex=False)
                      .astype(float))

In [ ]:
# Remoção de colunas não analizáveis
df=df.drop(["img_link", "product_link"], axis=1)

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 1465 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_id           1210 non-null   str    
 1   product_name         1465 non-null   str    
 2   category             1210 non-null   str    
 3   discounted_price     1210 non-null   float64
 4   actual_price         1210 non-null   float64
 5   discount_percentage  1210 non-null   float64
 6   rating               1210 non-null   float64
 7   rating_count         1465 non-null   float64
 8   about_product        1465 non-null   str    
 9   user_id              1210 non-null   str    
 10  user_name            1210 non-null   str    
 11  review_id            1210 non-null   str    
 12  review_title         1465 non-null   str    
 13  review_content       1465 non-null   str    
dtypes: float64(5), str(9)
memory usage: 204.0 KB


## Export para PowerBI

In [ ]:
df.to_csv("output\\amazon_cleaned.csv", sep=",")